# Gikomba Stall Analysis: Wanjiku's Fresh-Produce Business

**Analyst:** YOUR FULL NAME | **Cohort:** YOUR COHORT

Wanjiku runs a fresh-produce stall at Gikomba Market and has recorded 14 days of sales plus a rough stock list. This notebook answers: *How is the business doing? Is it growing? What should she restock, and where should she focus?* Only core Python is used (no pandas or NumPy).

## Data
The dataset exactly as provided.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Total, average, highest and lowest daily sales
I use `sum`, `len`, `max` and `min` for the headline numbers. `list.index()` gives the position of the best and worst days. Positions 0-6 are week 1, so the week number is `position // 7 + 1`.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_pos = daily_sales.index(highest)
worst_pos = daily_sales.index(lowest)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales   : KES {total_sales:,}")
print(f"Average daily : KES {average_sales:,.2f}")
print(f"Best day      : {best_day}, week {best_week} (KES {highest:,})")
print(f"Worst day     : {worst_day}, week {worst_week} (KES {lowest:,})")

Total sales   : KES 46,450
Average daily : KES 3,317.86
Best day      : Sat, week 2 (KES 5,600)
Worst day     : Tue, week 1 (KES 1,950)


## Task 2: Week-on-week growth
`week_totals()` slices the list into two weeks and returns both totals as a tuple. Growth is (week 2 - week 1) / week 1 x 100.

In [3]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) from a list of 14 daily sales."""
    return sum(sales[:7]), sum(sales[7:14])


week1, week2 = week_totals(daily_sales)
growth_pct = (week2 - week1) / week1 * 100

print(f"Week 1 total: KES {week1:,}")
print(f"Week 2 total: KES {week2:,}")
print(f"Week-on-week growth: {growth_pct:.2f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.66%


## Task 3: Classifying each day
`classify_day()` compares a day's sales with the average: more than 20% above is Strong, more than 20% below is Weak, otherwise Normal. A dictionary counting pattern (`.get(key, 0) + 1`) tallies the categories.

In [4]:
def classify_day(amount, average):
    """Return 'Strong' if amount is more than 20% above average,
    'Weak' if more than 20% below average, otherwise 'Normal'."""
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    return "Normal"


day_counts = {}
for day, amount in zip(days, daily_sales):
    label = classify_day(amount, average_sales)
    day_counts[label] = day_counts.get(label, 0) + 1

print(day_counts)

{'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays vs weekends
I loop over day names and amounts together, separating Mon-Fri from Sat-Sun, then compare the averages.

In [5]:
weekend_names = ("Sat", "Sun")
weekday_sales = [a for d, a in zip(days, daily_sales) if d not in weekend_names]
weekend_sales = [a for d, a in zip(days, daily_sales) if d in weekend_names]

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekends are {weekend_premium_pct:.1f}% higher than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends are 82.5% higher than weekdays


## Task 5: Reorder list and gross margins
`products_to_reorder()` loops through `stock.items()` and collects products at or below their reorder level. Gross margin % = (price - cost) / price x 100 for each product.

In [6]:
def products_to_reorder(stock_dict):
    """Return a list of product names whose qty is at or below the reorder level."""
    return [name for name, info in stock_dict.items() if info["qty"] <= info["reorder"]]


reorder_list = products_to_reorder(stock)

margins = {}
for name, info in stock.items():
    margins[name] = (info["price"] - info["cost"]) / info["price"] * 100

print("Reorder now:", reorder_list)
for name, m in margins.items():
    print(f"{name:<10} margin: {m:.1f}%")

Reorder now: ['Tomatoes', 'Sukuma']
Tomatoes   margin: 33.3%
Onions     margin: 30.0%
Sukuma     margin: 40.0%
Potatoes   margin: 33.3%
Avocado    margin: 42.9%


## Task 6: Stall Performance Report
An f-string report with fixed-width columns so numbers line up, and thousands separators on all KES amounts.

In [7]:
line = "=" * 44
print(line)
print("        GIKOMBA STALL PERFORMANCE REPORT")
print(line)
print("SALES SUMMARY (14 days)")
print(f"  {'Total sales':<22}KES {total_sales:>10,}")
print(f"  {'Average daily sales':<22}KES {average_sales:>10,.2f}")
print(f"  {'Best day':<22}{best_day} wk{best_week}, KES {highest:,}")
print(f"  {'Worst day':<22}{worst_day} wk{worst_week}, KES {lowest:,}")
print()
print("GROWTH")
print(f"  {'Week 1 total':<22}KES {week1:>10,}")
print(f"  {'Week 2 total':<22}KES {week2:>10,}")
print(f"  {'Week-on-week growth':<22}{growth_pct:>+13.2f}%")
print()
print("DAY CLASSIFICATION")
for label in ("Strong", "Normal", "Weak"):
    print(f"  {label:<22}{day_counts.get(label, 0):>3} days")
print()
print("WEEKDAY vs WEEKEND")
print(f"  {'Weekday average':<22}KES {weekday_avg:>10,.2f}")
print(f"  {'Weekend average':<22}KES {weekend_avg:>10,.2f}")
print(f"  {'Weekend premium':<22}{weekend_premium_pct:>13.1f}%")
print()
print("STOCK & MARGINS")
print(f"  {'Product':<10}{'Qty':>5}{'Reorder':>9}{'Margin':>9}  Status")
for name, info in stock.items():
    status = "REORDER" if name in reorder_list else "OK"
    print(f"  {name:<10}{info['qty']:>5}{info['reorder']:>9}{margins[name]:>8.1f}%  {status}")
print(line)

        GIKOMBA STALL PERFORMANCE REPORT
SALES SUMMARY (14 days)
  Total sales           KES     46,450
  Average daily sales   KES   3,317.86
  Best day              Sat wk2, KES 5,600
  Worst day             Tue wk1, KES 1,950

GROWTH
  Week 1 total          KES     22,050
  Week 2 total          KES     24,400
  Week-on-week growth          +10.66%

DAY CLASSIFICATION
  Strong                  4 days
  Normal                  4 days
  Weak                    6 days

WEEKDAY vs WEEKEND
  Weekday average       KES   2,685.00
  Weekend average       KES   4,900.00
  Weekend premium                82.5%

STOCK & MARGINS
  Product     Qty  Reorder   Margin  Status
  Tomatoes     18       25    33.3%  REORDER
  Onions       40       20    30.0%  OK
  Sukuma        9       15    40.0%  REORDER
  Potatoes     55       30    33.3%  OK
  Avocado      12       10    42.9%  OK


## Recommendations for Wanjiku

1. **Business is growing:** week 2 sales (KES 24,400) were 10.66% higher than week 1 (KES 22,050), so keep doing what is working.
2. **Focus on weekends:** Saturday and Sunday average KES 4,900 a day versus KES 2,685 on weekdays (82.5% higher), and all 4 "Strong" days were weekends, so stock up heavily on Friday evening and keep the stall fully staffed on Saturday and Sunday.
3. **Restock Tomatoes and Sukuma now:** Tomatoes are 7 kg below their reorder level (18 kg vs 25 kg) and Sukuma is 6 kg below (9 kg vs 15 kg), and both are fast-moving staples, so top them up before the weekend.
4. **Push Avocado and Sukuma for profit:** Avocado has the best margin (42.9%, KES 60 profit per kg) and Sukuma follows at 40.0%, so promote them at the front of the stall.
5. **Fix the weak weekdays:** 6 of 14 days were "Weak" (below KES 2,654), all of them weekdays, so try a Tuesday or Thursday offer (Tue week 1 was the worst day at KES 1,950) and avoid over-buying Potatoes (55 kg vs a 30 kg reorder level) and Onions (40 kg vs 20 kg) that tie up cash.